<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/EDA_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

Cloning into 'ML_fx'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 36 (delta 5), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 8.95 MiB | 11.68 MiB/s, done.
Resolving deltas: 100% (5/5), done.


# Extract Data

In [3]:
import pandas as pd
import numpy as np
import os
import zipfile
import io


def extract_data():
    RAW_DATA_DIR = "./ML_fx/data/raw/walmart-recruiting-store-sales-forecasting"

    print("Loading data...")

    stores_path = os.path.join(RAW_DATA_DIR, "stores.csv")
    if os.path.exists(stores_path):
        stores = pd.read_csv(stores_path)
    else:
        raise FileNotFoundError("stores.csv not found in raw directory.")

    train = pd.read_csv(os.path.join(RAW_DATA_DIR, "train.csv"))
    features = pd.read_csv(os.path.join(RAW_DATA_DIR, "features.csv"))
    test = pd.read_csv(os.path.join(RAW_DATA_DIR, "test.csv"))

    print(f"Train shape: {train.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Stores shape: {stores.shape}")
    print(f"Test shape: {test.shape}")
    return  train,  features, stores, test
train,  features, stores, test= extract_data()
print(train.head())
print("-------------------------")
print(train.describe())


Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
   Store  Dept        Date  Weekly_Sales  IsHoliday
0      1     1  2010-02-05      24924.50      False
1      1     1  2010-02-12      46039.49       True
2      1     1  2010-02-19      41595.55      False
3      1     1  2010-02-26      19403.54      False
4      1     1  2010-03-05      21827.90      False
-------------------------
               Store           Dept   Weekly_Sales
count  421570.000000  421570.000000  421570.000000
mean       22.200546      44.260317   15981.258123
std        12.785297      30.492054   22711.183519
min         1.000000       1.000000   -4988.940000
25%        11.000000      18.000000    2079.650000
50%        22.000000      37.000000    7612.030000
75%        33.000000      74.000000   20205.852500
max        45.000000      99.000000  693099.360000


# Take a look at the tables
let's see if there're negative sales numbers. also mean WeekleSales on Holiday and without it. plus let's specifically check on christmass
---



In [4]:
count_negative_sales = (train['Weekly_Sales'] < 0).sum()
print(count_negative_sales)

1285


In [9]:
avg_sales_by_holiday = train.groupby('IsHoliday')['Weekly_Sales'].mean()
print(avg_sales_by_holiday)
# 1. Ensure Date is datetime (if not already done)
train['Date'] = pd.to_datetime(train['Date'])

# 2. Define the date range
start_date = '2012-12-20' # Adjust year if needed, or use generic logic below
end_date = '2013-01-05'

# Generic approach that works for any year in your dataset:
# Filter for days where month/day falls between Dec 20 and Jan 5
mask_dec = (train['Date'].dt.month == 12) & (train['Date'].dt.day >= 20)
mask_jan = (train['Date'].dt.month == 1) & (train['Date'].dt.day <= 5)
date_mask = mask_dec | mask_jan

# 3. Filter for Holidays AND that specific date range
holiday_season_data = train[(train['IsHoliday'] == True) & date_mask]

# 4. Calculate Average
avg_holiday_season_sales = holiday_season_data['Weekly_Sales'].mean()

print(f"Average Sales during Holiday Season (Dec 20-Jan 5): ${avg_holiday_season_sales:,.2f}")



IsHoliday
False    15901.445069
True     17035.823187
Name: Weekly_Sales, dtype: float64
Average Sales during Holiday Season (Dec 20-Jan 5): $14,543.39


In [6]:
print(stores.head())
print("-------------------------")
print(stores.describe())

   Store Type    Size
0      1    A  151315
1      2    A  202307
2      3    B   37392
3      4    A  205863
4      5    B   34875
-------------------------
           Store           Size
count  45.000000      45.000000
mean   23.000000  130287.600000
std    13.133926   63825.271991
min     1.000000   34875.000000
25%    12.000000   70713.000000
50%    23.000000  126512.000000
75%    34.000000  202307.000000
max    45.000000  219622.000000


In [ ]:
print(features.head())
print("---------------------------")
print(features.describe())

   Store        Date  Temperature  Fuel_Price  MarkDown1  MarkDown2  \
0      1  2010-02-05        42.31       2.572        NaN        NaN   
1      1  2010-02-12        38.51       2.548        NaN        NaN   
2      1  2010-02-19        39.93       2.514        NaN        NaN   
3      1  2010-02-26        46.63       2.561        NaN        NaN   
4      1  2010-03-05        46.50       2.625        NaN        NaN   

   MarkDown3  MarkDown4  MarkDown5         CPI  Unemployment  IsHoliday  
0        NaN        NaN        NaN  211.096358         8.106      False  
1        NaN        NaN        NaN  211.242170         8.106       True  
2        NaN        NaN        NaN  211.289143         8.106      False  
3        NaN        NaN        NaN  211.319643         8.106      False  
4        NaN        NaN        NaN  211.350143         8.106      False  
             Store  Temperature   Fuel_Price      MarkDown1      MarkDown2  \
count  8190.000000  8190.000000  8190.000000    403

# Merge the tables
left join over store and dept

In [7]:
def merge_data(train, features, stores, test):
    print("Merging datasets...")

    # Merge train with features on Store and Date
    features=features.drop(columns=['IsHoliday'])
    df_train_full = train.merge(features, on=['Store', 'Date'], how='left')

    # Merge with stores info on Store ID
    df_train_full = df_train_full.merge(stores, on='Store', how='left')

    # Do the same for test set
    df_test_full = test.merge(features, on=['Store', 'Date'], how='left')
    df_test_full = df_test_full.merge(stores, on='Store', how='left')

    # Convert Date column to datetime
    df_train_full['Date'] = pd.to_datetime(df_train_full['Date'])
    df_test_full['Date'] = pd.to_datetime(df_test_full['Date'])
    return df_train_full, df_test_full

df_train_full, df_test_full=merge_data(train, features, stores, test)
print(df_train_full.head())
print("------------------------")
print(df_test_full.describe())

Merging datasets...
   Store  Dept       Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1 2010-02-05      24924.50      False        42.31       2.572   
1      1     1 2010-02-12      46039.49       True        38.51       2.548   
2      1     1 2010-02-19      41595.55      False        39.93       2.514   
3      1     1 2010-02-26      19403.54      False        46.63       2.561   
4      1     1 2010-03-05      21827.90      False        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5         CPI  \
0        NaN        NaN        NaN        NaN        NaN  211.096358   
1        NaN        NaN        NaN        NaN        NaN  211.242170   
2        NaN        NaN        NaN        NaN        NaN  211.289143   
3        NaN        NaN        NaN        NaN        NaN  211.319643   
4        NaN        NaN        NaN        NaN        NaN  211.350143   

   Unemployment Type    Size  
0         8.106    A  151315  
1         

In [8]:
avg_sales_by_type = df_train_full.groupby('Type')['Weekly_Sales'].mean()
print(avg_sales_by_type)

Type
A    20099.568043
B    12237.075977
C     9519.532538
Name: Weekly_Sales, dtype: float64


In [12]:
avg_sales_holiday_type = df_train_full.groupby(['IsHoliday', 'Type'])['Weekly_Sales'].mean()

# Print only the rows where IsHoliday is True
holiday_avgs = avg_sales_holiday_type.loc[True]

print(holiday_avgs)

Type
A    21297.517824
B    13346.164062
C     9532.963131
Name: Weekly_Sales, dtype: float64


In [ ]:
import os

# 1. Ensure the directory exists
output_dir = './ML_fx/data/processed'
os.makedirs(output_dir, exist_ok=True)

# 2. Save the DataFrame to CSV
df_train_full.to_csv(os.path.join(output_dir, 'train_processed.csv'), index=False)

print("File saved successfully!")


File saved successfully!


In [ ]:
df_test_full.to_csv(os.path.join(output_dir, 'test_processed.csv'), index=False)

print("File saved successfully!")

File saved successfully!
